In [30]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Smart MCQ Solver Challenge
## Deep Learning & Generative AI Course Project

### Student Details
## **Name: Varun Choudhary**

**Roll Number:** 24F2000187
---

## Project Overview

This project focuses on solving complex Multiple Choice Questions (MCQs) using Deep Learning and Transformer-based Natural Language Processing techniques.

The objective is to predict the top three most probable answer options among five candidate answers (A, B, C, D, and E). The project explores multiple approaches ranging from traditional NLP baselines to modern Transformer architectures, along with Retrieval-Augmented Generation (RAG) concepts and parameter-efficient fine-tuning.

The final model is evaluated using Mean Average Precision at 3 (MAP@3), the official metric of the Kaggle competition.

## Problem Statement

The Smart MCQ Solver Challenge aims to build an intelligent system capable of solving challenging multiple-choice questions.

Each sample contains

- A question prompt
- Five answer options (A–E)

The task is not only to identify the correct answer but to rank the three most likely answers in descending order of confidence.

The competition is evaluated using Mean Average Precision at 3 (MAP@3), rewarding models that rank the correct answer as high as possible.

---

## Objectives

- Explore the dataset through exploratory data analysis.
- Build multiple NLP models for answer prediction.
- Fine-tune Transformer-based language models.
- Compare multiple architectures using Weights & Biases.
- Improve ranking performance through experimentation and ensembling.

##  Libraries

In [31]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer
from sentence_transformers import SentenceTransformer

pd.set_option('display.max_colwidth', 120)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cuda


##  Load Dataset

In [32]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("Train.shape", train.shape)
print("Test.shape", test.shape)
print("Sample_submission.shape", sample_submission.shape)

Train.shape (2000, 8)
Test.shape (500, 7)
Sample_submission.shape (500, 2)


## Dataset Overview

In [33]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence?...,Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beg...,"Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past...",Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The rel...,Martin Heidegger believes that the relationship between time and human existence is cyclical. The past and present a...,"Martin Heidegger believes that time is an illusion, and the past, present, and future are all happening simultaneous...",B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energi...,Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energi...,Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energi...,Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energi...,Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energi...,A
2,3,Determine the correct option: What is the term used in astrophysics to describe light-matter interactions resulting ...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Martin Heidegger's view on the relationship between time and human existenc...,Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beg...,"Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past...",Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The rel...,Martin Heidegger believes that the relationship between time and human existence is cyclical. The past and present a...,"Martin Heidegger believes that time is an illusion, and the past, present, and future are all happening simultaneous...",B
4,5,"Identify the correct statement: What is the concept of simultaneity in Einstein's book, Relativity? carefully.","Simultaneity is relative, meaning that two events that appear simultaneous to an observer in a particular inertial r...","Simultaneity is relative, meaning that two events that appear simultaneous to an observer in a particular inertial r...","Simultaneity is absolute, meaning that two events that appear simultaneous to an observer in a particular inertial r...",Simultaneity is a concept that applies only to Newtonian theories and not to relativistic theories.,Simultaneity is a concept that applies only to relativistic theories and not to Newtonian theories.,A


In [34]:
train.tail()

,id,prompt,A,B,C,D,E,answer
1995,1996,What is the piezoelectric strain coefficient for AT-cut quartz crystals?,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1996,1997,Identify the correct statement: What is the synapstor or synapse transistor? based on the given context.,A device used to demonstrate a neuro-inspired circuit that shows short-term potentiation for learning and inactivity...,A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and activity-ba...,A device used to demonstrate a neuro-inspired circuit that shows short-term depression for learning and inactivity-b...,A device used to demonstrate a neuro-inspired circuit that shows short-term potentiation for learning and activity-b...,A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and inactivity-...,E
1997,1998,Determine the correct option: What does Earnshaw's theorem state? from the following choices.,A collection of point charges can be maintained in a stable stationary equilibrium configuration solely by the gravi...,A collection of point charges can be maintained in a stable stationary equilibrium configuration solely by the elect...,A collection of point charges can be maintained in a stable stationary equilibrium configuration solely by the magne...,A collection of point charges cannot be maintained in a stable stationary equilibrium configuration solely by the el...,A collection of point charges can be maintained in a stable stationary equilibrium configuration solely by the magne...,D
1998,1999,Identify the correct statement: What is the revised view of the atmosphere's nature based on the time-varying multis...,The atmosphere is a mechanism that is only influenced by large-scale processes and does not exhibit any small-scale ...,"The atmosphere possesses both chaos and order, including emerging organized mechanisms and time-varying forcing from...",The atmosphere is a structure that is only influenced by small-scale processes and does not exhibit any large-scale ...,The atmosphere is a completely chaotic mechanism with no order or organization.,The atmosphere is a completely ordered structure with no chaos or randomness.,B
1999,2000,Pick the best possible answer: Which of the following statements accurately describes the characteristics of gravita...,"Gravitational waves have an amplitude denoted by h, which represents the size of the wave. The amplitude varies with...","Gravitational waves have an amplitude denoted by λ, which represents the distance between points of maximum stretch ...","Gravitational waves have an amplitude denoted by h, which represents the size of the wave. The amplitude varies with...","Gravitational waves have an amplitude denoted by f, which represents the frequency of the wave's oscillation. The am...","Gravitational waves have an amplitude denoted by f, which represents the frequency of the wave's oscillation. The am...",C


In [35]:
train.describe()

,id
count,2000.000000
mean,1000.500000
std,577.494589
min,1.000000
25%,500.750000
50%,1000.500000
75%,1500.250000
max,2000.000000


In [36]:
print("\nTrain columns:", train.columns.tolist())


Train columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


In [37]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [38]:
train.nunique()

id        2000
prompt    1758
A          316
B          328
C          303
D          318
E          320
answer       5
dtype: int64

In [39]:
!pip install wandb

In [40]:
import wandb

wandb.login()

True

In [41]:
print('Missing values (train):\n', train.isnull().sum())
print()
print('Answer distribution:\n', train['answer'].value_counts())

Missing values (train):
 id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

Answer distribution:
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


In [42]:
OPTIONS = ['A', 'B', 'C', 'D', 'E']

len_records = []
for _, r in train.iterrows():
    for opt in OPTIONS:
        len_records.append({
            'is_correct': r['answer'] == opt,
            'opt_len': len(str(r[opt])),
        })
len_df = pd.DataFrame(len_records)
print(len_df.groupby('is_correct')['opt_len'].describe())

             count        mean         std  min    25%    50%    75%    max
is_correct                                                                 
False       8000.0  160.712375  106.093594  1.0   84.0  140.5  221.0  662.0
True        2000.0  181.879000  111.241740  2.0  102.0  183.0  253.0  587.0


In [43]:
# Text-reuse / leakage diagnostic
all_correct, all_wrong = [], []
for _, r in train.iterrows():
    for opt in OPTIONS:
        t = str(r[opt]).strip()
        (all_correct if r['answer'] == opt else all_wrong).append(t)

print('Correct texts:', len(all_correct), '| unique:', len(set(all_correct)))
print('Wrong texts:', len(all_wrong), '| unique:', len(set(all_wrong)))
print('Texts that are correct in one question but wrong in another:',
      len(set(all_correct) & set(all_wrong)))

Correct texts: 2000 | unique: 313
Wrong texts: 8000 | unique: 1271
Texts that are correct in one question but wrong in another: 0


In [44]:
from sklearn.model_selection import GroupShuffleSplit

# Build connected components: questions sharing any identical option text
# must land on the same side of the split, else the model can memorize
# the exact text instead of reasoning.
parent = {qid: qid for qid in train['id']}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[rx] = ry

text_to_first_qid = {}
for _, r in train.iterrows():
    qid = r['id']
    for opt in OPTIONS:
        t = str(r[opt]).strip()
        if t in text_to_first_qid:
            union(qid, text_to_first_qid[t])
        else:
            text_to_first_qid[t] = qid

train['group'] = train['id'].apply(find)
print('questions:', train['id'].nunique(), '| leak-free groups:', train['group'].nunique())

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(train, groups=train['group']))
train_df = train.iloc[train_idx].reset_index(drop=True)
val_df = train.iloc[val_idx].reset_index(drop=True)

print('train_df:', train_df.shape, '| val_df:', val_df.shape)
print('answer balance (train):\n', train_df['answer'].value_counts(normalize=True))
print('answer balance (val):\n', val_df['answer'].value_counts(normalize=True))

questions: 2000 | leak-free groups: 225
train_df: (1653, 9) | val_df: (347, 9)
answer balance (train):
 answer
B    0.254688
C    0.221416
A    0.201452
E    0.170599
D    0.151845
Name: proportion, dtype: float64
answer balance (val):
 answer
D    0.308357
C    0.268012
B    0.198847
E    0.121037
A    0.103746
Name: proportion, dtype: float64


In [45]:
def to_long_format(df, has_label=True):
    rows = []
    for _, r in df.iterrows():
        for opt in OPTIONS:
            row = {
                'id': r['id'],
                'option': opt,
                'prompt': str(r['prompt']),
                'option_text': str(r[opt]),
                'text': str(r['prompt']) + ' [SEP] ' + str(r[opt]),
            }
            if has_label:
                row['label'] = 1 if r['answer'] == opt else 0
            rows.append(row)
    return pd.DataFrame(rows)

train_long = to_long_format(train_df, has_label=True)
val_long = to_long_format(val_df, has_label=True)
test_long = to_long_format(test, has_label=False)

print('train_long:', train_long.shape, '| val_long:', val_long.shape, '| test_long:', test_long.shape)
train_long.head(5)

train_long: (8265, 6) | val_long: (1735, 6) | test_long: (2500, 5)


,id,option,prompt,option_text,text,label
0,1,A,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence?...,Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beg...,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence?...,0
1,1,B,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence?...,"Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past...",Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence?...,1
2,1,C,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence?...,Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The rel...,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence?...,0
3,1,D,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence?...,Martin Heidegger believes that the relationship between time and human existence is cyclical. The past and present a...,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence?...,0
4,1,E,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence?...,"Martin Heidegger believes that time is an illusion, and the past, present, and future are all happening simultaneous...",Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence?...,0


In [46]:
def map_at_3(scored_long_df, answer_lookup):
    total, n = 0.0, 0
    for qid, grp in scored_long_df.groupby('id'):
        ranked = grp.sort_values('score', ascending=False)['option'].tolist()[:3]
        true_ans = answer_lookup[qid]
        if true_ans in ranked:
            total += 1.0 / (ranked.index(true_ans) + 1)
        n += 1
    return total / n

val_answer_lookup = dict(zip(val_df['id'], val_df['answer']))

In [47]:
run1 = wandb.init(project='mcq-solver-challenge', name='model1-tfidf-logreg', reinit=True)

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)
X_train = tfidf.fit_transform(train_long['text'])
X_val = tfidf.transform(val_long['text'])
y_train = train_long['label']
y_val = val_long['label']

clf_tfidf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf_tfidf.fit(X_train, y_train)

val_long['score_m1'] = clf_tfidf.predict_proba(X_val)[:, 1]
val_pred_label = (val_long['score_m1'] >= 0.5).astype(int)

m1_map3 = map_at_3(val_long.rename(columns={'score_m1': 'score'}), val_answer_lookup)
m1_acc = accuracy_score(y_val, val_pred_label)
m1_f1 = f1_score(y_val, val_pred_label)

print(f'Model 1 (TF-IDF+LogReg) — MAP@3: {m1_map3:.4f} | Acc: {m1_acc:.4f} | F1: {m1_f1:.4f}')
wandb.log({'map@3': m1_map3, 'accuracy': m1_acc, 'f1': m1_f1})
run1.finish()

Model 1 (TF-IDF+LogReg) — MAP@3: 0.6652 | Acc: 0.7787 | F1: 0.2163


accuracy,▁
f1,▁
map@3,▁
accuracy,0.77867
f1,0.21633
map@3,0.66523


In [48]:
MODEL_NAME = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MCQPairDataset(Dataset):
    def __init__(self, df, tokenizer, has_label=True, max_len=256):
        self.prompts = df['prompt'].tolist()
        self.options = df['option_text'].tolist()
        self.has_label = has_label
        if has_label:
            self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.prompts[idx], self.options[idx],
            truncation=True, max_length=self.max_len, padding='max_length',
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        if self.has_label:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

train_ds_m2 = MCQPairDataset(train_long, tokenizer, has_label=True)
val_ds_m2 = MCQPairDataset(val_long, tokenizer, has_label=True)

model_m2 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight      

In [49]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

2.10.0+cu128
12.8
True Tesla T4


In [50]:
!pip install -q transformers sentence-transformers accelerate --no-deps
!pip install -q tokenizers safetensors huggingface_hub  # inke deps chahiye but torch nahi

In [51]:
training_args = TrainingArguments(
    output_dir='./deberta_mcq',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=50,
    report_to='wandb',
    run_name='model2-deberta-v3-base',
    fp16=False,   
)

trainer = Trainer(
    model=model_m2,
    args=training_args,
    train_dataset=train_ds_m2,
    eval_dataset=val_ds_m2,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan


TrainOutput(global_step=1551, training_loss=21.38610422711154, metrics={'train_runtime': 456.8103, 'train_samples_per_second': 54.279, 'train_steps_per_second': 3.395, 'total_flos': 3261977807569920.0, 'train_loss': 21.38610422711154, 'epoch': 3.0})

In [52]:
run2 = wandb.init(project='mcq-solver-challenge', name='model2-deberta-eval', reinit=True)

val_preds = trainer.predict(val_ds_m2)
val_probs = torch.softmax(torch.tensor(val_preds.predictions), dim=1)[:, 1].numpy()
val_long['score_m2'] = val_probs
val_pred_label_m2 = (val_long['score_m2'] >= 0.5).astype(int)

m2_map3 = map_at_3(val_long.rename(columns={'score_m2': 'score'}), val_answer_lookup)
m2_acc = accuracy_score(y_val, val_pred_label_m2)
m2_f1 = f1_score(y_val, val_pred_label_m2)

print(f'Model 2 (DeBERTa-v3) — MAP@3: {m2_map3:.4f} | Acc: {m2_acc:.4f} | F1: {m2_f1:.4f}')
wandb.log({'map@3': m2_map3, 'accuracy': m2_acc, 'f1': m2_f1})
run2.finish()

eval/runtime,▄█▁
eval/samples_per_second,▅▁█
eval/steps_per_second,▅▁█
train/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇█████
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇█████
train/learning_rate,███▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁
train/loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+2,...
eval/loss,nan
eval/runtime,7.8825
eval/samples_per_second,220.107


Model 2 (DeBERTa-v3) — MAP@3: 0.2925 | Acc: 0.8000 | F1: 0.0000


accuracy,▁
f1,▁
map@3,▁
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
+1,...
accuracy,0.8
f1,0
map@3,0.29251
test/loss,nan


In [53]:
sbert = SentenceTransformer('all-mpnet-base-v2', device=DEVICE)

def build_sbert_features(df):
    prompt_emb = sbert.encode(df['prompt'].tolist(), batch_size=64, show_progress_bar=True)
    option_emb = sbert.encode(df['option_text'].tolist(), batch_size=64, show_progress_bar=True)
    cos_sim = np.sum(prompt_emb * option_emb, axis=1) / (
        np.linalg.norm(prompt_emb, axis=1) * np.linalg.norm(option_emb, axis=1) + 1e-8
    )
    abs_diff = np.abs(prompt_emb - option_emb)
    features = np.hstack([cos_sim.reshape(-1, 1), abs_diff])
    return features

X_train_sbert = build_sbert_features(train_long)
X_val_sbert = build_sbert_features(val_long)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/130 [00:00<?, ?it/s]

Batches:   0%|          | 0/130 [00:00<?, ?it/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

In [54]:
run3 = wandb.init(project='mcq-solver-challenge', name='model3-sbert-logreg', reinit=True)

clf_sbert = LogisticRegression(max_iter=1000, class_weight='balanced')
clf_sbert.fit(X_train_sbert, y_train)

val_long['score_m3'] = clf_sbert.predict_proba(X_val_sbert)[:, 1]
val_pred_label_m3 = (val_long['score_m3'] >= 0.5).astype(int)

m3_map3 = map_at_3(val_long.rename(columns={'score_m3': 'score'}), val_answer_lookup)
m3_acc = accuracy_score(y_val, val_pred_label_m3)
m3_f1 = f1_score(y_val, val_pred_label_m3)

print(f'Model 3 (SBERT+LogReg) — MAP@3: {m3_map3:.4f} | Acc: {m3_acc:.4f} | F1: {m3_f1:.4f}')
wandb.log({'map@3': m3_map3, 'accuracy': m3_acc, 'f1': m3_f1})
run3.finish()

Model 3 (SBERT+LogReg) — MAP@3: 0.4870 | Acc: 0.6680 | F1: 0.2924


accuracy,▁
f1,▁
map@3,▁
accuracy,0.66801
f1,0.29238
map@3,0.48703


In [55]:
def normalize_per_question(df, score_col):
    return df.groupby('id')[score_col].transform(lambda s: (s - s.min()) / (s.max() - s.min() + 1e-8))

val_long['norm_m1'] = normalize_per_question(val_long, 'score_m1')
val_long['norm_m2'] = normalize_per_question(val_long, 'score_m2')
val_long['norm_m3'] = normalize_per_question(val_long, 'score_m3')
val_long['score_ensemble'] = (val_long['norm_m1'] + val_long['norm_m2'] + val_long['norm_m3']) / 3

ens_map3 = map_at_3(val_long.rename(columns={'score_ensemble': 'score'}), val_answer_lookup)
print(f'Ensemble MAP@3: {ens_map3:.4f}')
print(f'  vs Model 1: {m1_map3:.4f} | Model 2: {m2_map3:.4f} | Model 3: {m3_map3:.4f}')

Ensemble MAP@3: 0.2925
  vs Model 1: 0.6652 | Model 2: 0.2925 | Model 3: 0.4870


In [56]:
# refit TF-IDF+LogReg on full train (cheap — swap for the winning model/ensemble refit as needed)
full_long = to_long_format(train, has_label=True)
tfidf_full = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)
X_full = tfidf_full.fit_transform(full_long['text'])
clf_full = LogisticRegression(max_iter=1000, class_weight='balanced')
clf_full.fit(X_full, full_long['label'])

X_test = tfidf_full.transform(test_long['text'])
test_long['score'] = clf_full.predict_proba(X_test)[:, 1]

In [57]:
submission_rows = []
for qid, grp in test_long.groupby('id'):
    top3 = grp.sort_values('score', ascending=False)['option'].tolist()[:3]
    submission_rows.append({'ID': qid, 'Prediction': ' '.join(top3)})

submission = pd.DataFrame(submission_rows)
submission = submission.sort_values('ID').reset_index(drop=True)
submission.to_csv('submission.csv', index=False)
submission.head()

,ID,Prediction
0,1,A E C
1,2,B E A
2,3,B E D
3,4,E C A
4,5,C A B
